# 00 — Model Training: 20 Specialist Experiments

**Purpose:** Train 1 baseline + 19 augmented Mask R-CNN (ResNet-50-FPN) models using MMDetection.

> **Note:** Training outputs are not included. This notebook documents the exact configuration
> used for each experiment. All models share identical hyperparameters; only the augmentation
> pipeline differs.

**Hardware:** NVIDIA GeForce RTX 4080 | PyTorch 2.0.1 | CUDA 11.8

In [ ]:
import os
import copy
from mmengine.config import Config
from mmengine.runner import Runner

## Base Configuration

Load the MMDetection config and set project-specific parameters.
All 20 experiments inherit from this template.

In [ ]:
# --- Paths (adjust to your environment) ---
cfg_path = "configs/mask-rcnn_r50_fpn_1x_coco.py"
pretrained_weights = "weights/mask_rcnn_r50_fpn_1x_coco_20200205-d4b0c5d6.pth"
data_root = "data/"

# --- Load and customize config ---
cfg = Config.fromfile(cfg_path)
cfg.data_root = data_root
cfg.load_from = pretrained_weights
cfg.num_classes = 2
cfg.metainfo = {'classes': ('ac', 'lc')}

# Modify model heads for 2 classes
cfg.model.roi_head.bbox_head.num_classes = 2
cfg.model.roi_head.mask_head.num_classes = 2

# Configure data loaders
def update_data_cfg(data_cfg, split):
    data_cfg.dataset.data_root = data_root
    data_cfg.dataset.ann_file = f'{split}_annotations.json'
    data_cfg.dataset.data_prefix = dict(img=f'{split}/images/')
    data_cfg.dataset.metainfo = cfg.metainfo

update_data_cfg(cfg.train_dataloader, 'train')
update_data_cfg(cfg.val_dataloader, 'val')
update_data_cfg(cfg.test_dataloader, 'test')

cfg.val_evaluator.ann_file = os.path.join(data_root, 'val_annotations.json')
cfg.test_evaluator.ann_file = os.path.join(data_root, 'test_annotations.json')

# Baseline pipeline: NO augmentations
cfg.train_pipeline = [
    dict(type='LoadImageFromFile', backend_args=cfg.backend_args),
    dict(type='LoadAnnotations', with_bbox=True, with_mask=True),
    dict(type='Resize', scale=(1333, 800), keep_ratio=True),
    dict(type='PackDetInputs')
]
cfg.train_dataloader.dataset.pipeline = cfg.train_pipeline

# Training parameters
cfg.train_dataloader.batch_size = 2
cfg.optim_wrapper.optimizer.lr = 0.0025
cfg.train_cfg.max_epochs = 100
cfg.train_cfg.val_interval = 1
cfg.default_hooks.checkpoint = dict(
    type='CheckpointHook', interval=1,
    save_best='coco/bbox_mAP', rule='greater',
)

cfg_baseline_template = copy.deepcopy(cfg)

## Experiment Definitions

Each experiment deep-copies the baseline template and inserts one augmentation.
Augmentations use either native MMDetection transforms or the Albumentations wrapper.

### Spatial Mapping: Geometric

In [ ]:
# --- Exp 01: Baseline (No Augmentations) ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp01_baseline'
runner = Runner.from_cfg(cfg_exp)
runner.train()

In [ ]:
# --- Exp 02: HorizontalFlip ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp02_hflip'
cfg_exp.train_pipeline.insert(3, dict(type='RandomFlip', prob=0.5, direction='horizontal'))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

In [ ]:
# --- Exp 03: VerticalFlip ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp03_vflip'
cfg_exp.train_pipeline.insert(3, dict(type='RandomFlip', prob=0.5, direction='vertical'))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

In [ ]:
# --- Exp 04: Rotate (Albumentations) ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp04_rotate'
albu = dict(
    type='Albu',
    transforms=[dict(type='Rotate', limit=90, p=0.5)],
    bbox_params=dict(type='BboxParams', format='pascal_voc',
                     label_fields=['gt_bboxes_labels', 'gt_ignore_flags']),
    keymap={'img': 'image', 'gt_masks': 'masks', 'gt_bboxes': 'bboxes'}
)
cfg_exp.train_pipeline.insert(2, albu)
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

In [ ]:
# --- Exp 05: RandomScale ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp05_scale'
cfg_exp.train_pipeline[2] = dict(type='Resize', scale=(1333, 800),
                                  keep_ratio=True, scale_factor=(0.8, 1.2))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

In [ ]:
# --- Exp 06: Affine ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp06_affine'
albu = dict(
    type='Albu',
    transforms=[dict(type='Affine', scale=(0.8, 1.2),
                     translate_percent=(-0.1, 0.1), rotate=(-15, 15),
                     shear=(-10, 10), p=0.5)],
    bbox_params=dict(type='BboxParams', format='pascal_voc',
                     label_fields=['gt_bboxes_labels', 'gt_ignore_flags']),
    keymap={'img': 'image', 'gt_masks': 'masks', 'gt_bboxes': 'bboxes'}
)
cfg_exp.train_pipeline.insert(2, albu)
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

In [ ]:
# --- Exp 07: RandomResize + RandomCrop ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp07_randomresize_randomcrop'
cfg_exp.train_pipeline[2:3] = [
    dict(type='RandomResize', scale=(1333, 800), ratio_range=(0.5, 2.0), keep_ratio=True),
    dict(type='RandomCrop', crop_size=(800, 1333))
]
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

### Spatial Mapping: Distortion

In [ ]:
# --- Exp 12: GridDistortion ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp12_griddistortion'
cfg_exp.optim_wrapper.type = 'AmpOptimWrapper'
albu = dict(
    type='Albu',
    transforms=[dict(type='GridDistortion', p=0.5)],
    bbox_params=dict(type='BboxParams', format='pascal_voc',
                     label_fields=['gt_bboxes_labels', 'gt_ignore_flags']),
    keymap={'img': 'image', 'gt_masks': 'masks', 'gt_bboxes': 'bboxes'}
)
cfg_exp.train_pipeline.insert(2, albu)
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

In [ ]:
# --- Exp 13: OpticalDistortion ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp13_opticaldistortion'
cfg_exp.optim_wrapper.type = 'AmpOptimWrapper'
albu = dict(
    type='Albu',
    transforms=[dict(type='OpticalDistortion', p=0.5)],
    bbox_params=dict(type='BboxParams', format='pascal_voc',
                     label_fields=['gt_bboxes_labels', 'gt_ignore_flags']),
    keymap={'img': 'image', 'gt_masks': 'masks', 'gt_bboxes': 'bboxes'}
)
cfg_exp.train_pipeline.insert(2, albu)
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

### Pixel Forces: Photometric

In [ ]:
# Helper: Albumentations wrapper for pixel-level transforms
def make_albu_pixel(transform_dict):
    return dict(
        type='Albu', transforms=[transform_dict],
        bbox_params=dict(type='BboxParams', format='pascal_voc',
                         label_fields=['gt_bboxes_labels', 'gt_ignore_flags']),
        keymap={'img': 'image', 'gt_masks': 'masks', 'gt_bboxes': 'bboxes'}
    )

In [ ]:
# --- Exp 08: RandomBrightnessContrast ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp08_brightnesscontrast'
cfg_exp.train_pipeline.insert(2, make_albu_pixel(dict(type='RandomBrightnessContrast', p=0.5)))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

# --- Exp 09: HueSaturationValue ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp09_hsv'
cfg_exp.train_pipeline.insert(2, make_albu_pixel(dict(type='HueSaturationValue', p=0.5)))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

# --- Exp 10: CLAHE ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp10_clahe'
cfg_exp.optim_wrapper.type = 'AmpOptimWrapper'
cfg_exp.train_pipeline.insert(2, make_albu_pixel(dict(type='CLAHE', p=0.5)))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

# --- Exp 11: ChannelShuffle ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp11_channelshuffle'
cfg_exp.optim_wrapper.type = 'AmpOptimWrapper'
cfg_exp.train_pipeline.insert(2, make_albu_pixel(dict(type='ChannelShuffle', p=0.5)))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

### Pixel Forces: Blur

In [ ]:
# Helper: Albumentations wrapper for image-only transforms (no bbox mapping needed)
def make_albu_imgonly(transform_dict):
    return dict(type='Albu', transforms=[transform_dict], keymap={'img': 'image'})

In [ ]:
# --- Exp 18: MotionBlur ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp18_motionblur'
cfg_exp.optim_wrapper.type = 'AmpOptimWrapper'
cfg_exp.train_pipeline.insert(2, make_albu_imgonly(dict(type='MotionBlur', blur_limit=(3, 11), p=0.5)))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

# --- Exp 19: GaussianBlur ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp19_gaussianblur'
cfg_exp.optim_wrapper.type = 'AmpOptimWrapper'
cfg_exp.train_pipeline.insert(2, make_albu_imgonly(dict(type='GaussianBlur', blur_limit=(3, 7), p=0.5)))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

# --- Exp 20: MedianBlur ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp20_medianblur'
cfg_exp.optim_wrapper.type = 'AmpOptimWrapper'
cfg_exp.train_pipeline.insert(2, make_albu_imgonly(dict(type='MedianBlur', blur_limit=(3, 7), p=0.5)))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

### Pixel Forces: Noise

In [ ]:
# --- Exp 17: GaussNoise ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp17_gaussnoise'
cfg_exp.optim_wrapper.type = 'AmpOptimWrapper'
cfg_exp.train_pipeline.insert(2, make_albu_imgonly(dict(type='GaussNoise', p=0.5)))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

### Occlusion: Erasing

In [ ]:
# --- Exp 14: CoarseDropout (RandomErasing) ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp14_coarsedropout'
cfg_exp.optim_wrapper.type = 'AmpOptimWrapper'
cfg_exp.train_pipeline.insert(2, make_albu_imgonly(
    dict(type='CoarseDropout', p=0.5, max_holes=8,
         max_height=80, max_width=133, min_holes=1,
         min_height=20, min_width=20, fill_value=0)
))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

# --- Exp 15: Cutout ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp15_cutout'
cfg_exp.optim_wrapper.type = 'AmpOptimWrapper'
mean_vals = cfg_exp.model.data_preprocessor.mean
cfg_exp.train_pipeline.insert(2, make_albu_imgonly(
    dict(type='CoarseDropout', max_holes=8, max_height=32,
         max_width=32, fill_value=mean_vals, p=0.5)
))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()

# --- Exp 16: GridDropout ---
cfg_exp = copy.deepcopy(cfg_baseline_template)
cfg_exp.work_dir = './work_dirs/exp16_griddropout'
cfg_exp.optim_wrapper.type = 'AmpOptimWrapper'
cfg_exp.train_pipeline.insert(2, make_albu_imgonly(dict(type='GridDropout', p=0.5)))
cfg_exp.train_dataloader.dataset.pipeline = cfg_exp.train_pipeline
Runner.from_cfg(cfg_exp).train()